# Model 1 — logistic regression

This notebook trains **only Model 1** on the public UCI term-deposit subscription task. Its business analogue is STADIOEquities first-deposit activation; this data cannot model dormancy. Preprocessing and engineered features are shared with the other model for a fair comparison. See `Model1.MD`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "shared.py").exists(), "Open this notebook from the project root or its Model1 folder"
sys.path.insert(0, str(ROOT))
from shared import load_bank_data, chronological_splits, BankFeatureBuilder, train_one, score_one
from Model1.model1 import NAME, make_classifier

DATA_PATH = ROOT / "bank-full.csv"  # edit if your UCI CSV is elsewhere
OUTPUT_PATH = ROOT / "results" / "model1"
frame = load_bank_data(DATA_PATH if DATA_PATH.is_file() else None)
print(frame.shape, frame["y"].value_counts().to_dict())

## Inspect the shared preprocessing inputs

`duration` is excluded because it is known after the phone call. The `pdays=-1` sentinel is converted into a prior-contact indicator and an optional elapsed-days value.

In [ ]:
train, validation, test = chronological_splits(frame)
features = BankFeatureBuilder().fit_transform(train.drop(columns="y"))
print("Split sizes:", len(train), len(validation), len(test))
print("Model inputs:", features.columns.tolist())
print(features[["previously_contacted", "pdays_known", "total_contacts", "signed_log_balance"]].head())

## Train and evaluate Model 1

The classifier and its preprocessing pipeline fit on the earliest 60% of rows. The next 20% select an F2 threshold, and the last 20% are held out for final evaluation. Files are saved to `results/model1/`.

In [ ]:
results = train_one(DATA_PATH if DATA_PATH.is_file() else None, OUTPUT_PATH, NAME, make_classifier())
print("Test metrics:", results["test"])

## Optional inference on new UCI-shaped rows

Put a semicolon-separated `new_customers.csv` at the project root. The target and call duration columns can be omitted. This cell scores it only when present.

In [ ]:
NEW_DATA_PATH = ROOT / "new_customers.csv"
if NEW_DATA_PATH.is_file():
    score_one(NEW_DATA_PATH, OUTPUT_PATH, NAME)
else:
    print("Add new_customers.csv to run inference.")